### 1. Import Libraries

In [25]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


### 2. Read CSV Files

In [26]:
ges_degree_details = pd.read_csv("Data/ges_degree_details.csv")
ges_employment_outcomes = pd.read_csv("Data/ges_employment_outcomes.csv")
employment_by_industry = pd.read_csv("Data/employment_by_industry.csv")
retrenchment_by_industry = pd.read_csv("Data/retrenchment_by_industry.csv")
job_vacancies_by_industry = pd.read_csv("Data/job_vacancies_by_industry.csv")




### 3. Show First 5 Rows Of Datasets

In [27]:
ges_degree_details.head()

,year,university,school,degree,Key
0,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy and Business,1
1,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy (3-yr direct Honours Programme),2
2,2013,Nanyang Technological University,College of Business (Nanyang Business School),Business (3-yr direct Honours Programme),3
3,2013,Nanyang Technological University,College of Business (Nanyang Business School),Business and Computing,4
4,2013,Nanyang Technological University,College of Engineering,Aerospace Engineering,5


In [28]:
ges_employment_outcomes.head()

,employment_rate_overall,employment_rate_ft_perm,basic_monthly_mean,basic_monthly_median,gross_monthly_mean,gross_monthly_median,gross_mthly_25_percentile,gross_mthly_75_percentile,gross_monthly_mean.1,gross_monthly_median.1,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Key
0,97.4,96.1,3701,3200,3727,3350,2900,4000,3727,3350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,97.1,95.7,2850,2700,2938,2700,2700,2900,2938,2700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,90.9,85.7,3053,3000,3214,3000,2700,3500,3214,3000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,87.5,87.5,3557,3400,3615,3400,3000,4100,3615,3400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,95.3,95.3,3494,3500,3536,3500,3100,3816,3536,3500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5


In [29]:
employment_by_industry.head()

,year,sex,industry1,industry2,employment_status,employed
0,2010,male,manufacturing,manufacturing,employers,10800
1,2010,male,manufacturing,manufacturing,employees,170500
2,2010,male,manufacturing,manufacturing,own account workers,4400
3,2010,male,manufacturing,manufacturing,contributing family workers,400
4,2010,male,construction,construction,employers,11000


In [30]:
retrenchment_by_industry.head()

,year,industry,incidence_of_retrenchment
0,1998,manufacturing,62.9
1,1998,"food, beverages and tobacco",15.4
2,1998,textile and wearing apparel,31.1
3,1998,paper products and publishing,27
4,1998,petroleum and chemical products,30.1


In [31]:
job_vacancies_by_industry.head()

,year,industry,occupation,job_vacancy_rate
0,1998,manufacturing,"professionals, managers, executives and techni...",1.8
1,1998,manufacturing,"clerical, sales and services workers",1.2
2,1998,manufacturing,"production and transport operators, cleaners a...",1.6
3,1998,"food, beverages and tobacco","professionals, managers, executives and techni...",-
4,1998,"food, beverages and tobacco","clerical, sales and services workers",1.3


### 4. Data Cleaning

#### 4.1. Code to make a copy of ges_degree_details dataframe for cleaning

In [32]:
ges_degree_details_clean = ges_degree_details.copy()
print("Shape:", ges_degree_details_clean.shape)

Shape: (5135, 5)


In [33]:
column_summary = pd.DataFrame({
    "data_type": ges_degree_details_clean.dtypes.astype(str),
    "missing_count": (
        ges_degree_details_clean.isna().sum()
        ),

    "missing_percentage": (
        ges_degree_details_clean.isna().mean() * 100
    ).round(2),
    "unique_count": (
        ges_degree_details_clean.nunique(dropna=False)
    )
})

display(column_summary)

print("Empty Rows:", ges_degree_details_clean.isna().all(axis=1).sum())
print("Exact duplicate rows:", ges_degree_details_clean.duplicated().sum())
print("Duplicate keys:", ges_degree_details_clean["Key"].duplicated().sum())

,data_type,missing_count,missing_percentage,unique_count
year,str,0,0.0,12
university,str,0,0.0,7
school,str,0,0.0,71
degree,str,0,0.0,349
Key,int64,0,0.0,5135


Empty Rows: 0
Exact duplicate rows: 0
Duplicate keys: 0


In [34]:
placeholder_values = {
    "",
    "na",
    "n/a",
    "null",
    "none"
}

placeholder_report = []

for column in ["university", "school", "degree"]:
    normalised_values = (
        ges_degree_details_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    counts = normalised_values[
        normalised_values.isin(placeholder_values)
        ].value_counts()
    
    for value, count in counts.items():
        placeholder_report.append({
            "column": column,
            "placeholder_value": value,
            "count": count
        })

placeholder_report = pd.DataFrame(placeholder_report)

display(placeholder_report)

,column,placeholder_value,count
0,school,na,128


In [35]:
for column in ["university", "school", "degree"]:
    normalised_values = (
        ges_degree_details_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    placeholder_mask = normalised_values.isin(placeholder_values)

    ges_degree_details_clean.loc[
        placeholder_mask,
        column 
    ] = pd.NA

In [36]:
print("Missing values after converting placeholders:")
display(ges_degree_details_clean.isna().sum())

Missing values after converting placeholders:


year            0
university      0
school        128
degree          0
Key             0
dtype: int64

In [37]:
year_counts = (
    ges_degree_details_clean["year"].value_counts(dropna=False).sort_index()
)
display(year_counts)

year
2012     86
2013    340
2014    436
2015    492
2016    484
2017    528
2018    548
2019    560
2020    556
2021    540
2022    564
year      1
Name: count, dtype: int64

In [38]:
year_numeric_test = pd.to_numeric(
    ges_degree_details_clean["year"],
    errors="coerce"
)

In [39]:
invalid_year_mask = (
    year_numeric_test.isna()
    & ges_degree_details_clean["year"].notna()
)

print("Rows containing invalid year values:", invalid_year_mask.sum())

display(ges_degree_details_clean.loc[
    invalid_year_mask
])

Rows containing invalid year values: 1


,year,university,school,degree,Key
1262,year,university,school,degree,1263


In [40]:
rows_before = len(ges_degree_details_clean)

ges_degree_details_clean = (
    ges_degree_details_clean.loc[~invalid_year_mask].copy()
)

rows_removed = (
    rows_before - len(ges_degree_details_clean)
)

print("Rows removed due to invalid year values:", rows_removed)

Rows removed due to invalid year values: 1


In [41]:
ges_degree_details_clean["year"] = pd.to_numeric(
    ges_degree_details_clean["year"],
    errors = "coerce"
).astype("Int64")

In [42]:
print(
    ges_degree_details_clean["year"].dtype
)

display(
    ges_degree_details_clean["year"].value_counts(dropna=False).sort_index()
)

Int64


year
2012     86
2013    340
2014    436
2015    492
2016    484
2017    528
2018    548
2019    560
2020    556
2021    540
2022    564
Name: count, dtype: Int64

In [43]:
display(
    ges_degree_details_clean["university"].value_counts(dropna=False)
)

university
Nanyang Technological University                 1627
National University of Singapore                 1627
Singapore Institute of Technology                1136
Singapore Management University                   492
Singapore University of Technology and Design     128
Singapore University of Social Sciences           124
Name: count, dtype: int64

In [44]:
school_counts = (
    ges_degree_details_clean["school"].value_counts(dropna=False)
)

display(school_counts)

school
College of Engineering                                       669
College of Humanities, Arts & Social Sciences                376
Faculty of Engineering                                       274
Faculty of Science                                           222
School of Computing                                          218
                                                            ... 
Singapore Institute of Technology -Trinity College Dublin      4
YST Conservatory Of Music                                      4
SIT-DigiPen Institute of Technology                            4
SIT-Massey University                                          4
School of Law                                                  4
Name: count, Length: 70, dtype: int64

In [48]:
school_case_check = (
    ges_degree_details_clean
)

In [46]:
school_name_corrections = {
    "Faculty Of Dentistry": "Faculty of Dentistry",
    "Faculty Of Engineering": "Faculty of Engineering",
    "YST Conservatory Of Music": "YST Conservatory of Music"
}

ges_degree_details_clean["school"] = (
    ges_degree_details_clean["school"].replace(school_name_corrections)
)

In [47]:
display(
    ges_degree_details_clean["school"].value_counts(dropna=False)
)

school
College of Engineering                                       669
College of Humanities, Arts & Social Sciences                376
Faculty of Engineering                                       350
Faculty of Science                                           222
School of Computing                                          218
                                                            ... 
SIT-Trinity College Dublin                                     8
Singapore Institute of Technology -Trinity College Dublin      4
SIT-DigiPen Institute of Technology                            4
SIT-Massey University                                          4
School of Law                                                  4
Name: count, Length: 67, dtype: int64

In [49]:
display(
    ges_degree_details_clean["university"].value_counts(dropna=False)
)

university
Nanyang Technological University                 1627
National University of Singapore                 1627
Singapore Institute of Technology                1136
Singapore Management University                   492
Singapore University of Technology and Design     128
Singapore University of Social Sciences           124
Name: count, dtype: int64